# Optional: Zarr storage with xarray

This optional tutorial stores the deterministic temperature dataset from the introductory xarray tutorial in Zarr. The emphasis is not merely on calling `to_zarr`: we will choose storage chunks from expected access patterns and state what the small example cannot tell us about performance.

Allow approximately 30–40 minutes. Required background: xarray `Dataset` objects, named dimensions, and label- or position-based selection. Start JupyterLab from the repository root with `pixi run -e xarray lab`.

## Learning objectives and setup

After this tutorial, you should be able to:

- explain the difference between an xarray data model and a physical Zarr layout;
- write and verify a Zarr round trip;
- estimate chunk size and the number of chunks touched by simple access patterns;
- choose a defensible storage chunk shape for a stated workload; and
- distinguish storage-layout reasoning from performance claims that require representative measurements.

The notebook uses Zarr format 3 explicitly. Version-specific codec, cloud-store, distributed-write, and migration details are deliberately left to current documentation and performance-oriented training.

In [1]:
from math import ceil, prod
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import xarray as xr
import zarr

_ = xr.set_options(display_expand_attrs=False, display_expand_data=False)

## Recreate the evolving temperature dataset

The dataset is self-contained so that this optional notebook can be used independently. It represents a warm anomaly moving over a two-dimensional domain while spreading through diffusion. Its deterministic evolution makes every storage layout describe the same scientific values.

In [2]:
time = xr.date_range("2026-01-01", periods=49, freq="1h")
elapsed_hours = xr.DataArray(
    np.arange(time.size, dtype=float),
    dims="time",
    coords={"time": time},
    attrs={"long_name": "elapsed time", "units": "h"},
)
x_values = np.linspace(-5.0, 5.0, 81)
y_values = np.linspace(-5.0, 5.0, 81)
x = xr.DataArray(
    x_values,
    dims="x",
    coords={"x": x_values},
    attrs={"long_name": "eastward distance", "units": "km"},
)
y = xr.DataArray(
    y_values,
    dims="y",
    coords={"y": y_values},
    attrs={"long_name": "northward distance", "units": "km"},
)

In [3]:
mean_temperature = 15.0
diurnal_amplitude = 2.0
period = 24.0
initial_amplitude = 8.0
initial_width = 0.6
diffusivity = 0.015
initial_x, initial_y = -1.5, -0.8
velocity_x, velocity_y = 0.055, 0.025

ambient_temperature = (
    mean_temperature
    + diurnal_amplitude * np.sin(2.0 * np.pi * elapsed_hours / period)
).rename("ambient_temperature")
spread_variance = initial_width**2 + 2.0 * diffusivity * elapsed_hours
expected_x_center = (initial_x + velocity_x * elapsed_hours).rename(
    "expected_x_center"
)
expected_y_center = (initial_y + velocity_y * elapsed_hours).rename(
    "expected_y_center"
)
distance_squared = (x - expected_x_center) ** 2 + (y - expected_y_center) ** 2
temperature_anomaly = (
    initial_amplitude
    * initial_width**2
    / spread_variance
    * np.exp(-distance_squared / (2.0 * spread_variance))
).transpose("time", "y", "x")
temperature = (ambient_temperature + temperature_anomaly).rename("temperature")

In [4]:
temperature.attrs = {
    "long_name": "surface temperature",
    "units": "degC",
}
ambient_temperature.attrs = {
    "long_name": "ambient temperature",
    "units": "degC",
}
expected_x_center.attrs = {
    "long_name": "expected eastward anomaly centre",
    "units": "km",
}
expected_y_center.attrs = {
    "long_name": "expected northward anomaly centre",
    "units": "km",
}
dataset = xr.Dataset(
    data_vars={
        "temperature": temperature,
        "ambient_temperature": ambient_temperature,
        "expected_x_center": expected_x_center,
        "expected_y_center": expected_y_center,
    },
    coords={"elapsed_hours": elapsed_hours},
    attrs={
        "title": "Deterministic moving and diffusing temperature anomaly",
        "model": "Analytical advection-diffusion surrogate",
    },
)
dataset

<xarray.Dataset> Size: 3MB
Dimensions:              (time: 49, x: 81, y: 81)
Coordinates:
  * time                 (time) datetime64[ns] 392B 2026-01-01 ... 2026-01-03
    elapsed_hours        (time) float64 392B 0.0 1.0 2.0 3.0 ... 46.0 47.0 48.0
  * x                    (x) float64 648B -5.0 -4.875 -4.75 ... 4.75 4.875 5.0
  * y                    (y) float64 648B -5.0 -4.875 -4.75 ... 4.75 4.875 5.0
Data variables:
    temperature          (time, y, x) float64 3MB 15.0 15.0 15.0 ... 15.0 15.0
    ambient_temperature  (time) float64 392B 15.0 15.52 16.0 ... 14.0 14.48 15.0
    expected_x_center    (time) float64 392B -1.5 -1.445 -1.39 ... 1.085 1.14
    expected_y_center    (time) float64 392B -0.8 -0.775 -0.75 ... 0.375 0.4
Attributes: (2)

The three-dimensional temperature variable contains only about 2.5 MiB of raw floating-point values. That is large enough to illustrate chunk geometry but far too small for credible storage-performance benchmarks. Timings here would be dominated by fixed overhead, caching, and the local filesystem.

## A Zarr store and a verified round trip

An xarray `Dataset` describes variables, dimensions, coordinates, and attributes. Zarr determines how each array is divided into separately addressable chunks and how its metadata and encoded bytes are stored. On a local filesystem a store is represented by directories and files; on object storage, the same logical entries are objects rather than ordinary files.

We first let Zarr choose its default chunk layout. Defaults are useful for a smoke test, but they are not evidence that the layout fits a particular workload.

The `consolidated` parameter concerns **metadata only**: consolidation collects the metadata for all arrays and groups at the store root, reducing the number of metadata requests needed to discover a dataset. It neither combines data chunks nor changes their encoding. `None`, the xarray default, writes consolidated metadata and prefers it when inspecting an existing store, with a fallback to separate metadata; `True` explicitly requires consolidation; and `False` disables it. The same choice on `open_zarr` controls how metadata is read.

This notebook uses `consolidated=False` because consolidated metadata for Zarr format 3 is currently an experimental extension rather than part of the formal format specification. Disabling it avoids that interoperability caveat and its warning, but opening a remote store with many arrays may require more requests. For a read-heavy, slowly changing store used within the xarray/Zarr-Python ecosystem, consolidation can instead be beneficial; after structural metadata changes, it must be regenerated to avoid a stale snapshot.

In [5]:
temporary_directory = TemporaryDirectory()
store_root = Path(temporary_directory.name)
default_store = store_root / "temperature-default.zarr"

_ = dataset.to_zarr(
    default_store,
    mode="w",
    zarr_format=3,
    consolidated=False,
)

In [6]:
zarr_root = zarr.open_group(default_store, mode="r")
stored_temperature = zarr_root["temperature"]
{
    "shape": stored_temperature.shape,
    "chunk shape": stored_temperature.chunks,
    "dtype": str(stored_temperature.dtype),
    "format": stored_temperature.metadata.zarr_format,
}

{'shape': (49, 81, 81),
 'chunk shape': (25, 41, 41),
 'dtype': 'float64',
 'format': 3}

In [7]:
stored_paths = sorted(
    path.relative_to(default_store).as_posix()
    for path in default_store.rglob("*")
    if path.is_file()
)
{
    "number of local files": len(stored_paths),
    "first entries": stored_paths[:12],
}

{'number of local files': 24,
 'first entries': ['ambient_temperature/c/0',
  'ambient_temperature/zarr.json',
  'elapsed_hours/c/0',
  'elapsed_hours/zarr.json',
  'expected_x_center/c/0',
  'expected_x_center/zarr.json',
  'expected_y_center/c/0',
  'expected_y_center/zarr.json',
  'temperature/c/0/0/0',
  'temperature/c/0/0/1',
  'temperature/c/0/1/0',
  'temperature/c/0/1/1']}

The paths expose that Zarr is not one monolithic file. Their exact names are format and implementation details, so analysis code should use xarray or Zarr APIs rather than depend on this directory structure.

In [8]:
reopened = xr.open_zarr(
    default_store,
    chunks=None,
    consolidated=False,
).load()
xr.testing.assert_identical(dataset, reopened)
print("The default-layout store preserves values, coordinates, and attributes.")

The default-layout store preserves values, coordinates, and attributes.


Passing `chunks=None` tells xarray not to construct Dask arrays. The arrays still have storage chunks in Zarr; storage chunking and Dask computation chunking are related but distinct decisions. The latter belongs in a performance and scaling discussion.

## Choose chunks from access patterns

For a three-dimensional variable with chunk shape $(n_t,n_y,n_x)$, the maximum uncompressed bytes in one full chunk are

$$
B = n_t n_y n_x s.
$$

Here $n_t$, $n_y$, and $n_x$ are the chunk lengths along time, northward distance, and eastward distance, respectively; $s$ is the number of bytes per stored value; and $B$ is the maximum uncompressed chunk size in bytes. Boundary chunks can be smaller.

A chunk shape is a tuple of lengths, not a number of chunks. Reading any value generally requires reading and decoding its entire containing chunk.

In [9]:
temperature_shape = dataset["temperature"].shape
bytes_per_value = dataset["temperature"].dtype.itemsize

layouts = {
    "snapshot": (1, 81, 81),
    "time-series": (49, 9, 9),
    "balanced": (7, 27, 27),
}


def layout_summary(chunk_shape):
    chunk_grid = tuple(
        ceil(size / chunk)
        for size, chunk in zip(temperature_shape, chunk_shape, strict=True)
    )
    return {
        "chunk grid": chunk_grid,
        "temperature chunks": prod(chunk_grid),
        "maximum chunk KiB": prod(chunk_shape) * bytes_per_value / 1024.0,
        "chunks per snapshot": chunk_grid[1] * chunk_grid[2],
        "chunks per point time series": chunk_grid[0],
    }

In [10]:
for layout_name, chunk_shape in layouts.items():
    summary = layout_summary(chunk_shape)
    print(
        f"{layout_name:11} chunks={chunk_shape!s:13} "
        f"grid={summary['chunk grid']!s:11} "
        f"max={summary['maximum chunk KiB']:5.1f} KiB  "
        f"snapshot={summary['chunks per snapshot']:2d}  "
        f"point-series={summary['chunks per point time series']:2d}"
    )

snapshot    chunks=(1, 81, 81)   grid=(49, 1, 1)  max= 51.3 KiB  snapshot= 1  point-series=49
time-series chunks=(49, 9, 9)    grid=(1, 9, 9)   max= 31.0 KiB  snapshot=81  point-series= 1
balanced    chunks=(7, 27, 27)   grid=(7, 3, 3)   max= 39.9 KiB  snapshot= 9  point-series= 7


The snapshot-oriented layout reads one chunk for a complete field but 49 chunks for one complete point time series. The time-series-oriented layout reverses that trade-off. The balanced layout touches more than one chunk for either query but avoids the worst case of both specialised layouts.

These chunks are only tens of KiB because the teaching dataset is tiny. **Copying these numerical sizes into a large production store would create excessive metadata and per-chunk overhead.** What transfers is the reasoning about shape and access, not the literal numbers.

## Write explicit layouts

Xarray accepts storage chunks through the per-variable `encoding` passed to `to_zarr`. Chunking applies to coordinate arrays as well as data variables. Here every one-dimensional coordinate or auxiliary variable is stored as a single chunk, while the three-dimensional temperature variable uses the layout under investigation.

The positional chunk tuple must follow each variable's dimension order. Named dimensions make that order inspectable, but the encoding itself is positional.

In [11]:
def zarr_encoding(source_dataset, temperature_chunks):
    encoding = {
        variable_name: {"chunks": tuple(variable.shape)}
        for variable_name, variable in source_dataset.variables.items()
    }
    encoding["temperature"] = {"chunks": temperature_chunks}
    return encoding

In [12]:
layout_stores = {}
for layout_name, chunk_shape in layouts.items():
    store_path = store_root / f"temperature-{layout_name}.zarr"
    _ = dataset.to_zarr(
        store_path,
        mode="w",
        zarr_format=3,
        consolidated=False,
        encoding=zarr_encoding(dataset, chunk_shape),
    )
    layout_stores[layout_name] = store_path

for layout_name, store_path in layout_stores.items():
    zarr_array = zarr.open_group(store_path, mode="r")["temperature"]
    local_file_count = sum(path.is_file() for path in store_path.rglob("*"))
    print(
        f"{layout_name:11} stored chunks={zarr_array.chunks!s:13} "
        f"local files={local_file_count}"
    )

snapshot    stored chunks=(1, 81, 81)   local files=65
time-series stored chunks=(49, 9, 9)    local files=97
balanced    stored chunks=(7, 27, 27)   local files=79


The local file count includes metadata and chunks for every variable, not just temperature. Nevertheless, the comparison exposes a real design pressure: smaller independently readable chunks usually mean more storage objects. Object count can matter on both shared filesystems and object stores, although for different reasons.

In [13]:
for layout_name, store_path in layout_stores.items():
    roundtrip = xr.open_zarr(
        store_path,
        chunks=None,
        consolidated=False,
    ).load()
    xr.testing.assert_identical(dataset, roundtrip)

print("All three physical layouts preserve the same logical dataset.")

All three physical layouts preserve the same logical dataset.


## Investigation: choose a layout for a mixed workload

A downstream workflow makes nine independent complete-field queries and twenty independent complete point-time-series queries. Assume no cache reuse between queries and count only temperature chunks; ignore compression, metadata reads, and boundary chunks being smaller.

For each candidate layout, estimate the total number of chunk reads and choose a layout. Explain why this is a workload-specific decision rather than a universally optimal chunk shape. Predict the result before running the reference solution.

In [14]:
workload = {
    "complete snapshots": 9,
    "complete point time series": 20,
}
workload

{'complete snapshots': 9, 'complete point time series': 20}

### Reference solution

In [15]:
def estimated_chunk_reads(chunk_shape, snapshot_queries, point_series_queries):
    summary = layout_summary(chunk_shape)
    return (
        snapshot_queries * summary["chunks per snapshot"]
        + point_series_queries * summary["chunks per point time series"]
    )


estimated_reads = {
    layout_name: estimated_chunk_reads(
        chunk_shape,
        snapshot_queries=workload["complete snapshots"],
        point_series_queries=workload["complete point time series"],
    )
    for layout_name, chunk_shape in layouts.items()
}

for layout_name, chunk_reads in estimated_reads.items():
    print(f"{layout_name:11} estimated temperature-chunk reads: {chunk_reads}")

best_layout = min(estimated_reads, key=estimated_reads.get)
print(f"\nLowest estimate for this workload: {best_layout}")

snapshot    estimated temperature-chunk reads: 989
time-series estimated temperature-chunk reads: 749
balanced    estimated temperature-chunk reads: 221

Lowest estimate for this workload: balanced


Under this deliberately simple cost model, the balanced layout wins: it avoids the 49 separate time chunks required by each point-series query in the snapshot layout and the 81 spatial chunks required by each field query in the time-series layout.

This is not a performance benchmark. Real results also depend on cache reuse, query alignment, compression ratio and decoding cost, metadata latency, concurrency, storage technology, Dask task size, and whether multiple chunks are grouped into Zarr v3 shards. A different workload can—and should—produce a different choice.

## Optional: beyond this notebook

Topics that matter for large or shared datasets include:

- choosing compression codecs and levels from measured size/throughput trade-offs;
- aligning larger Dask computation chunks with storage chunks;
- measuring rechunking and distributed-write costs;
- using consolidated metadata when many metadata requests are expensive;
- coordinating appends or region writes safely; and
- using Zarr format 3 sharding to group many independently readable chunks into fewer storage objects.

Those choices require representative data, queries, worker memory, concurrency, and storage hardware. They belong in Python-for-HPC rather than being inferred from this small in-memory example.

Use the current [xarray I/O guide](https://docs.xarray.dev/en/stable/user-guide/io.html), [Zarr performance guide](https://zarr.readthedocs.io/en/stable/user-guide/performance/), and [Dask array chunking guidance](https://docs.dask.org/en/stable/array-chunks.html) when making production decisions; defaults and supported encodings evolve.

## Recap

- Xarray defines the logical labeled dataset; Zarr defines a chunked physical representation.
- Verify a round trip rather than assuming values and metadata were preserved.
- Choose storage chunks from anticipated selections and writes, then check chunk bytes and object count.
- Apply encoding to every variable that needs an intentional layout, including coordinates.
- Storage chunks and Dask chunks are distinct; computational chunks are often larger aligned multiples.
- Small demonstrations can validate structure and reasoning, but not production performance.

In [16]:
temporary_directory.cleanup()